<a href="https://colab.research.google.com/github/NidhiDekate/car-price-multimodal-dl/blob/main/08_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

base = '/content/drive/MyDrive/CarPricePrediction/'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

# load scaler and encoders from notebook 03
with open(base + 'data/tabular/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(base + 'data/tabular/encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

print("scaler loaded")
print("encoders loaded:", list(encoders.keys()))


Mounted at /content/drive
device: cuda
scaler loaded
encoders loaded: ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior', 'seller']


In [ ]:
from xgboost import XGBRegressor

# load best model
with open(base + 'models/xgb_best.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

print("XGBoost model loaded")

# test prediction with dummy input
dummy = np.zeros((1, 12))
pred  = xgb_model.predict(dummy)
print(f"test prediction: ${np.expm1(pred[0]):,.0f}")

XGBoost model loaded
test prediction: $12,843


In [ ]:
def predict_price(year, make, model_name, trim, body,
                  transmission, state, condition,
                  odometer, color, interior):

    # feature engineering - same as notebook 03
    car_age      = 2015 - int(year)
    log_odometer = np.log1p(float(odometer))

    # brand tier
    luxury     = ['Ferrari', 'Rolls-Royce', 'Bentley', 'Lamborghini', 'Aston Martin']
    premium    = ['Porsche', 'Jaguar', 'Bmw', 'Mercedes-Benz', 'Audi',
                  'Cadillac', 'Infiniti', 'Acura', 'Lexus']
    brand_tier = 2 if make in luxury else (1 if make in premium else 0)

    # encode categorical features
    def safe_encode(encoder, value, default=0):
        try:
            return encoder.transform([value])[0]
        except:
            return default

    make_enc         = safe_encode(encoders['make'],         make)
    model_enc        = safe_encode(encoders['model'],        model_name)
    trim_enc         = safe_encode(encoders['trim'],         trim)
    body_enc         = safe_encode(encoders['body'],         body)
    transmission_enc = safe_encode(encoders['transmission'], transmission)
    state_enc        = safe_encode(encoders['state'],        state)
    color_enc        = safe_encode(encoders['color'],        color)
    interior_enc     = safe_encode(encoders['interior'],     interior)

    # build feature vector - same order as notebook 03
    features = np.array([[
        car_age, log_odometer, float(condition), brand_tier,
        make_enc, model_enc, trim_enc, body_enc,
        transmission_enc, state_enc, color_enc, interior_enc
    ]])

    # scale
    features_scaled = scaler.transform(features)

    # predict
    log_price = xgb_model.predict(features_scaled)[0]
    price     = np.expm1(log_price)

    return round(price, 2)


# test it
test_price = predict_price(
    year=2012, make='Honda', model_name='Accord', trim='EX',
    body='Sedan', transmission='automatic', state='CA',
    condition=35, odometer=45000, color='white', interior='black'
)
print(f"test prediction: ${test_price:,.0f}")

test prediction: $13,016


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr

# dropdown options from encoders
makes         = sorted(list(encoders['make'].classes_))
bodies        = sorted(list(encoders['body'].classes_))
transmissions = sorted(list(encoders['transmission'].classes_))
states        = sorted(list(encoders['state'].classes_))
colors        = sorted(list(encoders['color'].classes_))
interiors     = sorted(list(encoders['interior'].classes_))

def gradio_predict(year, make, model_name, trim, body,
                   transmission, state, condition, odometer, color, interior):
    try:
        price = predict_price(
            year, make, model_name, trim, body,
            transmission, state, condition, odometer, color, interior
        )
        return f"${price:,.0f}"
    except Exception as e:
        return f"Error: {str(e)}"

# build interface
with gr.Blocks(title="Car Price Predictor") as app:
    gr.Markdown("# 🚗 Car Price Predictor")
    gr.Markdown("Enter car details to get predicted auction price. Model: XGBoost trained on 558K car sales.")

    with gr.Row():
        with gr.Column():
            year         = gr.Slider(2000, 2015, value=2012, step=1, label="Year")
            make         = gr.Dropdown(makes, value="Ford", label="Make")
            model_name   = gr.Textbox(value="F-150", label="Model")
            trim         = gr.Textbox(value="XLT", label="Trim")
            body         = gr.Dropdown(bodies, value="Sedan", label="Body Style")
            transmission = gr.Dropdown(transmissions, value="automatic", label="Transmission")

        with gr.Column():
            state     = gr.Dropdown(states, value="ca", label="State")
            condition = gr.Slider(1, 50, value=35, step=1, label="Condition (1-50, higher=better)")
            odometer  = gr.Number(value=45000, label="Odometer (miles)")
            color     = gr.Dropdown(colors, value="white", label="Exterior Color")
            interior  = gr.Dropdown(interiors, value="black", label="Interior Color")

    predict_btn = gr.Button("Predict Price", variant="primary")
    output      = gr.Textbox(label="Predicted Price")

    predict_btn.click(
        fn=gradio_predict,
        inputs=[year, make, model_name, trim, body,
                transmission, state, condition, odometer, color, interior],
        outputs=output
    )

    gr.Markdown("### Sample predictions")
    gr.Examples(
        examples=[
            [2012, "Honda", "Accord", "EX", "Sedan", "automatic", "ca", 35, 45000, "white", "black"],
            [2012, "Bmw", "3 Series", "328i", "Sedan", "automatic", "ny", 40, 30000, "black", "black"],
            [2008, "Ford", "F-150", "XLT", "Crew Cab", "automatic", "tx", 25, 85000, "silver", "gray"],
        ],
        inputs=[year, make, model_name, trim, body,
                transmission, state, condition, odometer, color, interior]
    )

print("app defined")

app defined


In [ ]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00486309ab91e3c0a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
